# IR WARRIORS: End-to-End Landsat IR Enhancement
This notebook uses real Landsat Collection 2 Level-2 scenes. It does not create synthetic metrics or model outputs.

In [ ]:
!git clone https://github.com/ravirajmore7/IR-colorization-and-interpritation.git
%cd IR-colorization-and-interpritation/ir-warriors-mvp
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Edit these paths to four extracted, real Landsat scene directories.
SCENES = [
    '/content/drive/MyDrive/landsat/scene_1',
    '/content/drive/MyDrive/landsat/scene_2',
    '/content/drive/MyDrive/landsat/scene_3',
    '/content/drive/MyDrive/landsat/scene_4',
]
assert all(__import__('pathlib').Path(path).is_dir() for path in SCENES), 'Update SCENES with extracted Landsat folders.'

In [ ]:
from training.prepare_dataset import generate_patches
manifest = generate_patches(SCENES, 'data/patches', patch_size=256, stride=192, min_valid_fraction=0.75)
print(manifest)

In [ ]:
from training.train import TrainConfig, train
result = train(TrainConfig(manifest=str(manifest), epochs=20, batch_size=4, workers=2))
result

In [ ]:
from app.services.inference_service import infer_manifest
evaluation = infer_manifest(manifest, 'checkpoints/best_generator.pth', 'outputs')
evaluation['metrics'], evaluation['warnings']

In [ ]:
import subprocess, sys
api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])
ui = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'frontend/streamlit_app.py', '--server.port', '8501'])
print('Services started on ports 8000 and 8501. Use Colab port forwarding or run the UI locally against saved Drive artifacts.')